# Benchmark performance analysis

This notebook summarizes the independent runs of each algorithm by their median performance on each benchmark problem. The benchmark problems are then used as paired blocks in the global statistical comparison.

# 1. Imports and experiment configuration

In [ ]:
# Adds the parent directory to sys.path
import sys
from pathlib import Path

sys.path.append('../')

import pickle

import numpy as np
import pandas as pd
from autorank import autorank
from problems.benchmark_problems import get_pareto
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

RESULTS_FOLDER = Path('../results')
ALPHA = 0.05
EXPECTED_NUM_RUNS = 30
HV_REFERENCE_VERTEX = 10
PARETO_DENSITY = 100
PARETO_RANDOM_SEED = 42

# Display name: filename prefix
ALGORITHMS = {
    'A-MESH': 'AMESH',
    'MACO': 'MACO',
    'MOPSO-CD': 'MOPSO_CD',
    'NSGA-II': 'NSGA2',
    'NSPSO': 'NSPSO',
    'SPEA2': 'SPEA2',
}

# Available problem configurations: number of objectives, decision variables and WFG k.
PROBLEM_CONFIGURATIONS = {
    'zdt1': (2, 20, None),
    'zdt2': (2, 20, None),
    'zdt3': (2, 20, None),
    'zdt4': (2, 20, None),
    'zdt6': (2, 20, None),
    'dtlz1': (3, 20, None),
    'dtlz2': (3, 20, None),
    'dtlz3': (3, 20, None),
    'dtlz4': (3, 20, None),
    'dtlz5': (3, 20, None),
    'dtlz6': (3, 20, None),
    'dtlz7': (3, 20, None),
    'wfg1': (3, 20, 10),
    'wfg2': (3, 20, 10),
    'wfg3': (3, 20, 10),
    'wfg4': (3, 20, 10),
    'wfg5': (3, 20, 10),
    'wfg6': (3, 20, 10),
    'wfg7': (3, 20, 10),
    'wfg8': (3, 20, 10),
    'wfg9': (3, 20, 10),
}

# Only the problems listed here enter the median tables and the paired tests.
# Remove or reorder names here to change the blocks compared by Autorank.
SELECTED_PROBLEMS = [
    'zdt1', 'zdt2', 'zdt3', 'zdt4', 'zdt6',
    'dtlz1', 'dtlz2', 'dtlz3', 'dtlz4', 'dtlz5', 'dtlz6', 'dtlz7',
    'wfg1', 'wfg2', 'wfg3', 'wfg4', 'wfg5', 'wfg6', 'wfg7', 'wfg8', 'wfg9',
]

if len(SELECTED_PROBLEMS) != len(set(SELECTED_PROBLEMS)):
    raise ValueError('SELECTED_PROBLEMS contains duplicate problem names.')

EXPERIMENTS = [
    (problem_name, *PROBLEM_CONFIGURATIONS[problem_name])
    for problem_name in SELECTED_PROBLEMS
]

# 2. Load the independent runs and calculate performance summaries

In [ ]:
def get_reproducible_pareto(problem_name, n_obj, n_var, wfg_k):
    # Pymoo samples the WFG reference fronts with default_rng(None). A shared
    # seeded generator keeps the IGD reference sets fixed across executions.
    original_default_rng = np.random.default_rng
    pareto_rng = original_default_rng(PARETO_RANDOM_SEED)

    def seeded_default_rng(seed=None):
        return pareto_rng if seed is None else original_default_rng(seed)

    np.random.default_rng = seeded_default_rng
    try:
        return get_pareto(
            problem_name, PARETO_DENSITY, n_var, n_obj, wfg_k
        )
    finally:
        np.random.default_rng = original_default_rng


def load_result(problem_name, n_obj, n_var, file_prefix):
    file_path = RESULTS_FOLDER / f'{file_prefix}_{problem_name}_{n_obj}_{n_var}.pkl'
    if not file_path.exists():
        raise FileNotFoundError(f'Result file not found: {file_path}')

    with file_path.open('rb') as file:
        return pickle.load(file)


def extract_run_fronts(result):
    # Integer keys identify the independent runs; the 'combined' entry is excluded.
    run_keys = sorted(key for key in result if isinstance(key, int))
    if len(run_keys) != EXPECTED_NUM_RUNS:
        raise ValueError(
            f'Expected {EXPECTED_NUM_RUNS} independent runs, but found {len(run_keys)}.'
        )
    return [result[key]['F'] for key in run_keys]


def summarize(values):
    values = np.asarray(values, dtype=float)
    return {
        'Mean': values.mean(),
        'Std. Dev.': values.std(ddof=1) if values.size > 1 else 0.0,
        'Median': np.median(values),
        'Min.': values.min(),
        'Max.': values.max(),
    }

In [ ]:
records = []

for problem_name, n_obj, n_var, wfg_k in EXPERIMENTS:
    hv_indicator = HV(ref_point=np.full(n_obj, HV_REFERENCE_VERTEX))
    pareto_front = get_reproducible_pareto(
        problem_name, n_obj, n_var, wfg_k
    )
    igd_indicator = IGD(pareto_front)

    for algorithm_name, file_prefix in ALGORITHMS.items():
        result = load_result(problem_name, n_obj, n_var, file_prefix)
        fronts = extract_run_fronts(result)
        hv_values = [hv_indicator(front) for front in fronts]
        igd_values = [igd_indicator(front) for front in fronts]

        hv_summary = summarize(hv_values)
        igd_summary = summarize(igd_values)
        records.append({
            'Problem': problem_name.upper(),
            'Algorithm': algorithm_name,
            'Runs': len(fronts),
            **{f'{name} HV': value for name, value in hv_summary.items()},
            **{f'{name} IGD': value for name, value in igd_summary.items()},
        })

summary_df = pd.DataFrame.from_records(records)
summary_df

# 3. Median performance tables

Each cell below is the median of 30 independent runs. Higher HV and lower IGD are preferable. These tables provide problem-level descriptive results; they do not contain per-problem significance tests.

In [ ]:
problem_order = [name.upper() for name in SELECTED_PROBLEMS]
algorithm_order = list(ALGORITHMS)


def build_problem_median_matrix(data, indicator):
    value_column = f'Median {indicator}'
    matrix = (
        data
        .pivot(index='Problem', columns='Algorithm', values=value_column)
        .reindex(index=problem_order, columns=algorithm_order)
    )

    expected_shape = (len(SELECTED_PROBLEMS), len(ALGORITHMS))
    if matrix.shape != expected_shape:
        raise ValueError(
            f'{indicator} median matrix has shape {matrix.shape}; '
            f'expected {expected_shape}.'
        )
    if matrix.isna().any().any():
        missing = matrix.isna().stack()
        missing = missing[missing].index.tolist()
        raise ValueError(f'Missing {indicator} medians for: {missing}')

    return matrix


# Each row is one benchmark problem; each entry is the median of 30 runs.
median_hv = build_problem_median_matrix(summary_df, 'HV')
median_igd = build_problem_median_matrix(summary_df, 'IGD')

display(median_hv)
display(median_igd)

In [ ]:
def format_median_table(table, maximize, decimals):
    rounded = table.round(decimals)
    formatted = rounded.copy().astype(object)
    for problem, row in rounded.iterrows():
        best = row.max() if maximize else row.min()
        for algorithm, value in row.items():
            text = f'{value:.{decimals}f}'
            formatted.loc[problem, algorithm] = (
                f'\\textbf{{{text}}}' if value == best else text
            )
    formatted.index.name = 'Problem'
    return formatted


formatted_hv = format_median_table(median_hv, maximize=True, decimals=3)
formatted_igd = format_median_table(median_igd, maximize=False, decimals=4)

print(formatted_hv.to_latex(
    escape=False,
    column_format='l' + 'r' * len(algorithm_order),
    caption='Median HV over 30 independent runs',
    label='tab:benchmark_hv_medians',
))
print(formatted_igd.to_latex(
    escape=False,
    column_format='l' + 'r' * len(algorithm_order),
    caption='Median IGD over 30 independent runs',
    label='tab:benchmark_igd_medians',
))

# 4. Paired comparison of the problem-level medians

The input to each hypothesis test is the corresponding median matrix constructed above: 21 selected benchmark problems (rows) by six algorithms (columns). Thus, each row is a paired block identified by the same problem across all algorithms, and each cell is the median of 30 independent executions. The raw executions themselves are not passed to Autorank. The non-parametric mode is forced because the comparison is rank-based and the indicator scales differ across problems. Autorank therefore applies the Friedman omnibus test followed, when appropriate, by the Nemenyi post-hoc procedure.

In [ ]:
# These are the exact 21 x 6 matrices passed to Autorank.
hv_problem_medians = median_hv.loc[problem_order, algorithm_order].copy()
igd_problem_medians = median_igd.loc[problem_order, algorithm_order].copy()

print(f'HV test input: {hv_problem_medians.shape[0]} problems x '
      f'{hv_problem_medians.shape[1]} algorithms')
print(f'IGD test input: {igd_problem_medians.shape[0]} problems x '
      f'{igd_problem_medians.shape[1]} algorithms')
print('Problems included:', ', '.join(hv_problem_medians.index))

autorank_report_hv = autorank(
    hv_problem_medians,
    order='descending',
    alpha=ALPHA,
    force_mode='nonparametric',
)
autorank_report_igd = autorank(
    igd_problem_medians,
    order='ascending',
    alpha=ALPHA,
    force_mode='nonparametric',
)

In [ ]:
def display_test_summary(indicator_name, report):
    print(f'{indicator_name}:')
    print(f'  Paired benchmark problems: {report.num_samples}')
    print(f'  Omnibus test: {report.omnibus}')
    print(f'  Omnibus p-value: {report.pvalue:.6g}')
    print(f'  Post-hoc procedure: {report.posthoc}')
    print(f'  Critical distance: {report.cd:.6g}')
    if report.pvalue < report.alpha:
        print('  Decision: reject the omnibus null hypothesis.')
        print('  The Nemenyi pairwise comparisons may be interpreted.')
    else:
        print('  Decision: do not reject the omnibus null hypothesis.')
        print('  The Nemenyi pairwise comparisons should not be interpreted.')


display_test_summary('HV', autorank_report_hv)
display_test_summary('IGD', autorank_report_igd)

In [ ]:
rank_summary = pd.DataFrame({
    'Algorithm': algorithm_order,
    'HV mean rank': [
        autorank_report_hv.rankdf.loc[name, 'meanrank']
        for name in algorithm_order
    ],
    'IGD mean rank': [
        autorank_report_igd.rankdf.loc[name, 'meanrank']
        for name in algorithm_order
    ],
}).set_index('Algorithm')

display(rank_summary.sort_values('HV mean rank'))

In [ ]:
formatted_ranks = rank_summary.copy().astype(object)
for column in rank_summary.columns:
    best = rank_summary[column].min()
    for algorithm, value in rank_summary[column].items():
        text = f'{value:.2f}'
        formatted_ranks.loc[algorithm, column] = (
            f'\\textbf{{{text}}}' if np.isclose(value, best) else text
        )

print(formatted_ranks.to_latex(
    escape=False,
    column_format='lrr',
    caption='Mean ranks across the benchmark problems',
    label='tab:benchmark_mean_ranks',
))

## Interpretation reminder

If the Friedman omnibus null hypothesis is rejected, two algorithms differ significantly under the Nemenyi procedure when the absolute difference between their mean ranks exceeds the reported critical distance. A difference that does not exceed the critical distance means that no significant difference was detected; it does not establish statistical equivalence.